# 05 — V3 Training (Heterogeneous graph + nutrient hubs)

v1 MVP에서 그래프 구조만 바꿈: K개 영양소 hub 노드 추가 + I-N 엣지 추가.
Model은 v1 GISMo 그대로 재사용 (use_health_goal=True). Hub 노드는 F 노드와 동일하게 random init embedding → WeightedGINConv로 propagation.

v2 (feature 인젝션) ↔ v3 (structural 인젝션) 의 온전한 isolation 비교.

T4 기준 약 2시간. 4개 g 조건 한 번에 평가됨.

**저장**: `{OUTPUT_DIR}/best_v3.pt`, `{OUTPUT_DIR}/test_predictions_v3_{{auto,1_0,0_1,1_1}}.json`

## 환경 설정

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q torch_geometric pandas numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
PROJECT_ROOT = '/content/drive/MyDrive/CS471_project'
CODE_DIR = f'{PROJECT_ROOT}/code'
DATA_DIR = f'{PROJECT_ROOT}/data'
OUTPUT_DIR = f'{PROJECT_ROOT}/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(CODE_DIR)
print(f'Working dir: {os.getcwd()}')

In [ ]:
import torch, torch_geometric
print(f'PyTorch: {torch.__version__}  PyG: {torch_geometric.__version__}  CUDA: {torch.cuda.is_available()}')

In [ ]:
import json
with open(f'{DATA_DIR}/data_meta.json') as f:
    META = json.load(f)
NUM_TOTAL = META['num_total_nodes']
NUM_ING   = META['num_ingredients']
print(f'NUM_TOTAL={NUM_TOTAL}  NUM_ING={NUM_ING}')
print(f'(v3 will add 7 hub nodes on top → effective total {NUM_TOTAL + 7})')

## 학습 + 평가

In [ ]:
# V3 학습 + 4개 g 조건 평가 (한 번에)
# Hub nutrients (7개): calorie_kcal, fat_g, saturated_fat_g, carbohydrate_g,
#                       sugar_g, protein_g, sodium_mg
# I-N edge weight: log1p + per-nutrient min-max → (0.01, 1]
#
# Base FlavorGraph edge weight 범위가 해제다른 scale면 --hub_weight_scale로 조절
# (학습 시작시 base/v3 edge weight range가 찍힌니 비교해보고 조정).
!python train_v3.py \
    --data_dir {DATA_DIR} \
    --output_dir {OUTPUT_DIR} \
    --num_total_nodes {NUM_TOTAL} --num_ingredients {NUM_ING} \
    --batch_size 64 --max_epochs 200 --patience 10 \
    --lambda_h 1.0 --margin 0.5 --tau_percentile 50 \
    --hub_weight_scale 1.0 \
    --test_g_overrides auto 1_0 0_1 1_1